# 03 · U-Net training — Tumour segmentation (Dice + BCE)
**Owner:** M3 (ML Engineer — Diagnosis) · **Drop 4**

**Data (pick one)**
* **A — M2's prepared pairs** (BraTS / LUNA16 slices): `images/` + `masks/` with identical file names.
* **B — backup:** *Add data →* `mateuszbuda/lgg-mri-segmentation` (110 patients, 3,929 FLAIR slices + masks).

**Outputs:** `models/best_unet.pth`, `results/unet_training_curves.png`, `results/segmentation_overlay.png`, `results/unet_test_metrics.json`

**Target:** test Dice ≥ 0.80

In [ ]:
# --- get the repo code into the Kaggle session ---------------------------------
# Option 1 (repo is private): upload the repo as a Kaggle *Dataset* named "medical-ai-code"
#   and attach it -> it appears under /kaggle/input/medical-ai-code
# Option 2: !git clone https://<TOKEN>@github.com/<team>/medical-ai.git
import os, sys, shutil, glob
CODE_SRC = next(iter(glob.glob("/kaggle/input/**/src/diagnosis", recursive=True)), None)
if CODE_SRC and not os.path.exists("/kaggle/working/medical-ai"):
    shutil.copytree(os.path.dirname(os.path.dirname(CODE_SRC)), "/kaggle/working/medical-ai")
REPO = "/kaggle/working/medical-ai" if os.path.exists("/kaggle/working/medical-ai") else os.path.abspath("..")
os.chdir(REPO); sys.path.insert(0, REPO)
print("repo:", REPO)

In [ ]:
!pip -q install albumentations opencv-python-headless 2>/dev/null | tail -1

In [ ]:
import torch
print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))   # screenshot this for the group (Drop 1)

## 1 · Data — image/mask pairs, patient-level split

In [ ]:
import glob
IMG_DIR, MASK_DIR = None, None          # Layout A, e.g. "/kaggle/input/brats-slices/images", ".../masks"
lgg = glob.glob("/kaggle/input/**/kaggle_3m", recursive=True)
SEG_ROOT = lgg[0] if (lgg and not IMG_DIR) else None   # Layout B
print("SEG_ROOT:", SEG_ROOT, "| IMG_DIR:", IMG_DIR)
DATA_ARGS = f'--seg_root "{SEG_ROOT}"' if SEG_ROOT else f'--img_dir "{IMG_DIR}" --mask_dir "{MASK_DIR}"'

from src.data.dataset import make_seg_loaders
loaders, df = make_seg_loaders(SEG_ROOT, IMG_DIR, MASK_DIR, batch_size=8, num_workers=2)
display(df.groupby("split").agg(images=("image", "size"), patients=("patient_id", "nunique")))

In [ ]:
# mask alignment check (Drop 4 hand-off from M2)
import matplotlib.pyplot as plt
from src.preprocessing.augment import denormalize
x, m = next(iter(loaders["train"]))
fig, ax = plt.subplots(2, 8, figsize=(18, 4.5))
for i in range(8):
    ax[0, i].imshow(denormalize(x[i])); ax[0, i].axis("off")
    ax[1, i].imshow(denormalize(x[i])); ax[1, i].imshow(m[i, 0], alpha=0.4, cmap="Reds"); ax[1, i].axis("off")
plt.show()
print("fraction of train slices with tumour:", float((m.flatten(1).sum(1) > 0).float().mean()))

## 2 · Train (25 epochs, AdamW 1e-3, cosine LR, Dice+BCE) — ~20–30 min on T4 for LGG

In [ ]:
!python -m src.diagnosis.train_unet $DATA_ARGS --epochs 25 --bs 16 --lr 1e-3 --out models/best_unet.pth

In [ ]:
from IPython.display import Image, display
display(Image("results/unet_training_curves.png"))

## 3 · Test Dice / IoU + overlays
`dice_mean` = mean per-slice Dice over **all** test slices (empty-predicted-on-empty counts as 1). `dice_tumour_slices` = mean Dice only on slices that contain tumour — report **both**; the second is the harder, more honest number.

In [ ]:
!python -m src.diagnosis.evaluate_unet --ckpt models/best_unet.pth $DATA_ARGS --out_dir results
import json; print(json.dumps(json.load(open("results/unet_test_metrics.json")), indent=2))
display(Image("results/segmentation_overlay.png"))

## 4 · Inference function used by the API (Drop 7)

In [ ]:
from src.diagnosis.inference import predict_segmentation
r = predict_segmentation(df[df.split == "test"].image.iloc[0])
print({k: v for k, v in r.items() if not k.endswith("base64")})

## 5 · Save artefacts

In [ ]:
!cd /kaggle/working && zip -qr m3_unet_outputs.zip medical-ai/models/best_unet.pth medical-ai/models/unet_history.json medical-ai/results && ls -lh m3_unet_outputs.zip

## 6 · Results (fill in after the run)
| Metric (test) | Value |
|---|---|
| Dice (all slices) | |
| Dice (tumour slices) | |
| IoU (all slices) | |
| Global Dice | |